# PPO Validation-Only Analysis

Paired patient/scenario analysis with 10,000 hierarchical bootstrap replicates. The held-out test cohort is not opened. Attention is not interpreted as a causal effect. Commands stream output live, emit idle heartbeats, and include at least the Last 200 lines in failure reports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo = Path('/content/VitalDB-Feature-Selection')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/khyitty/VitalDB-Feature-Selection.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
subprocess.run([
    'git', '-C', str(repo), 'merge-base', '--is-ancestor',
    '767f3bff3dcaeabc51049fba5ccba1ac02b69ae3', 'HEAD'
], check=True)
os.chdir(repo)
from src.colab_workflow import run_streaming_command as run_command
import pandas as pd
import torch
torch_before, pandas_before = torch.__version__, pd.__version__
dry_run = run_command(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '-r', 'requirements-rl.txt'],
    stage='RL dependency dry run',
)
proposal = dry_run.stdout
if 'Would install torch-' in proposal or 'Would install pandas-' in proposal:
    raise RuntimeError('RL dependency profile would replace torch or pandas; aborting.')
run_command([sys.executable, '-m', 'pip', 'install', 'stable-baselines3==2.9.0'], stage='RL dependency install')
import pandas as pd_after
import torch as torch_after
if torch_after.__version__ != torch_before or pd_after.__version__ != pandas_before:
    raise RuntimeError('torch/pandas changed during analysis dependency setup.')

In [ ]:
drive_project = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
dataset_dir = drive_project / 'data/modeling/full'
project_data_root = drive_project / 'data'
protocol_dir = drive_project / 'outputs/ppo_protocol'
output_root = drive_project / 'outputs/ppo_control_comparison'
analysis_dir = drive_project / 'outputs/ppo_validation_analysis'
access = json.loads((protocol_dir / 'cohort_access_manifest.json').read_text())
for key in ('test_trajectory_loaded', 'test_outcomes_evaluated', 'test_policy_rollout_performed'):
    if access[key] is not False:
        raise RuntimeError(f'Held-out test seal violated: {key}={access[key]!r}')
print('Drive modeling dataset:', dataset_dir)
print('Drive project data root:', project_data_root)
print('Demographics source:', access['selected_demographics_path'])
print('Validation analysis reads completed validation artifacts only; held-out test trajectory remains sealed.')
run_command([
    sys.executable, 'scripts/analyze_ppo_experiments.py',
    '--output-root', str(output_root),
    '--analysis-dir', str(analysis_dir),
    '--bootstrap-replicates', '10000',
    '--bootstrap-seed', '20260716',
], stage='validation-only PPO analysis')

In [ ]:
import pandas as pd
from IPython.display import display

summary = json.loads((analysis_dir / 'validation_analysis_summary.json').read_text())
bootstrap = pd.read_csv(analysis_dir / 'hierarchical_bootstrap_summary.csv')
attention = pd.read_csv(analysis_dir / 'attention_feature_summary.csv')
print(json.dumps(summary, indent=2))
display(bootstrap)
display(attention)

Validation analysis does not alter the frozen protocol and does not select a winner from a p-value. A separate locked workflow is required before any one-time held-out RL test.